In [1]:
import os
import random
import shutil
import zipfile
from pathlib import Path

# ---------------- Config ----------------
SRC_ROOT   = Path("/kaggle/input/datasets/ndkhoa240805/musan-geo-agla/musan")
DST_ROOT   = Path("/kaggle/working/musan_25")
ZIP_PATH   = Path("/kaggle/working/musan_25.zip")
RATIO      = 0.25
SEED       = 42
AUDIO_EXTS = {".wav"}          # thêm ".flac", ".mp3" nếu cần
COPY_META  = True              # copy README/LICENSE... ở root
# ----------------------------------------

rng = random.Random(SEED)
DST_ROOT.mkdir(parents=True, exist_ok=True)

manifest = []          # (relative_path, size_bytes)
n_total = n_kept = 0

# 1) Duyệt từng thư mục, lấy mẫu 25% file audio trong đó
for dirpath, _, filenames in os.walk(SRC_ROOT):
    dirpath = Path(dirpath)
    rel_dir = dirpath.relative_to(SRC_ROOT)

    audio = sorted(f for f in filenames if Path(f).suffix.lower() in AUDIO_EXTS)
    if audio:
        k = max(1, round(len(audio) * RATIO))   # tối thiểu 1 file/thư mục
        chosen = rng.sample(audio, k)
        (DST_ROOT / rel_dir).mkdir(parents=True, exist_ok=True)

        for fname in sorted(chosen):
            src, dst = dirpath / fname, DST_ROOT / rel_dir / fname
            shutil.copyfile(src, dst)           # xử lý từng file một
            manifest.append((str(rel_dir / fname), dst.stat().st_size))

        n_total += len(audio)
        n_kept  += len(chosen)

    # file không phải audio (README, LICENSE, ...) chỉ copy ở root
    if COPY_META and rel_dir == Path("."):
        for f in filenames:
            if Path(f).suffix.lower() not in AUDIO_EXTS:
                shutil.copyfile(dirpath / f, DST_ROOT / f)

# 2) Ghi manifest để đảm bảo tái lập
with open(DST_ROOT / "subset_manifest.tsv", "w") as fh:
    fh.write(f"# seed={SEED} ratio={RATIO}\n")
    for rel, size in manifest:
        fh.write(f"{rel}\t{size}\n")

print(f"Đã chọn {n_kept}/{n_total} file ({n_kept / max(n_total, 1):.2%})")

# 3) Zip folder output (WAV nén kém nên dùng deflate mức thấp cho nhanh)
if ZIP_PATH.exists():
    ZIP_PATH.unlink()
with zipfile.ZipFile(ZIP_PATH, "w", zipfile.ZIP_DEFLATED, compresslevel=1) as zf:
    for f in sorted(DST_ROOT.rglob("*")):
        if f.is_file():
            zf.write(f, arcname=Path("musan_25") / f.relative_to(DST_ROOT))

print(f"Zip: {ZIP_PATH} ({ZIP_PATH.stat().st_size / 1e9:.2f} GB)")

# (tuỳ chọn) xoá thư mục trung gian để tiết kiệm disk /kaggle/working
# shutil.rmtree(DST_ROOT)

Đã chọn 504/2016 file (25.00%)
Zip: /kaggle/working/musan_25.zip (2.73 GB)


In [2]:
!ls /kaggle/working/musan_25

music  noise  README  speech  subset_manifest.tsv
